<a href="https://colab.research.google.com/github/anaghahegde326/BIS_LAB/blob/main/BIS_LAB4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import math

# Levy flight
def levy_flight(size):
    beta = 1.5
    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta *
         2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)

    return u / (np.abs(v) ** (1 / beta))


# Starting and destination points
start = np.array([0.0, 0.0])
goal = np.array([10.0, 10.0])

# Circular obstacle
obstacle = np.array([5.0, 5.0])
radius = 1.5


# Calculate path length and obstacle penalty
def fitness(path):
    points = np.vstack([
        start, path.reshape(3, 2), goal
    ])

    distance = np.linalg.norm(
        np.diff(points, axis=0), axis=1
    ).sum()

    penalty = 0

    for a, b in zip(points[:-1], points[1:]):
        ab = b - a
        t = np.clip(
            np.dot(obstacle - a, ab) /
            (np.dot(ab, ab) + 1e-12), 0, 1
        )

        closest = a + t * ab
        d = np.linalg.norm(closest - obstacle)

        if d < radius:
            penalty += 1000 * (radius - d) ** 2

    return distance + penalty


# Cuckoo Search
def cuckoo_search():
    nests = 30
    iterations = 500
    pa = 0.25

    population = np.random.uniform(0, 10, (nests, 6))
    scores = np.array([fitness(x) for x in population])

    for _ in range(iterations):

        i = np.random.randint(nests)

        new_solution = (
            population[i] +
            0.02 * levy_flight(6) *
            (population[i] -
             population[np.random.randint(nests)])
        )

        new_solution = np.clip(new_solution, 0, 10)

        new_score = fitness(new_solution)

        j = np.random.randint(nests)

        if new_score < scores[j]:
            population[j] = new_solution
            scores[j] = new_score

        # Abandon worst nests
        abandon = np.random.rand(nests) < pa

        population[abandon] = np.random.uniform(
            0, 10, (abandon.sum(), 6)
        )

        scores[abandon] = np.array([
            fitness(x) for x in population[abandon]
        ])

    best = np.argmin(scores)

    return population[best], scores[best]


# Main program
np.random.seed(7)

best_path, best_distance = cuckoo_search()

print("Best UAV Path:")
print("Start:", start)

for point in best_path.reshape(3, 2):
    print("Waypoint:", np.round(point, 2))

print("Destination:", goal)
print("Optimized Path Cost:", round(best_distance, 2))


Best UAV Path:
Start: [0. 0.]
Waypoint: [4.96 1.57]
Waypoint: [3.92 2.15]
Waypoint: [6.3 2. ]
Destination: [10. 10.]
Optimized Path Cost: 17.59


In [2]:

import numpy as np
import math

# Levy flight
def levy_flight():
    beta = 1.5

    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta *
         2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)

    return u / (abs(v) ** (1 / beta))


# Robot settings
distance = 10
max_velocity = 2


# Fitness function
def fitness(T):
    maximum_speed = 15 / T

    penalty = max(0, maximum_speed - max_velocity)

    return T + 100000 * penalty ** 2


# Improved Cuckoo Search
def improved_cuckoo_search():

    nests = 20
    iterations = 300
    pa = 0.25

    population = np.random.uniform(1, 20, nests)
    scores = np.array([fitness(x) for x in population])

    best_index = np.argmin(scores)
    best = population[best_index]
    best_score = scores[best_index]

    for iteration in range(iterations):

        # Adaptive step size
        step = 0.15 * (1 - iteration / iterations) + 0.005

        for i in range(nests):

            new_solution = (
                population[i] +
                step * levy_flight() *
                (population[i] - best)
            )

            new_solution = np.clip(new_solution, 1, 20)

            new_score = fitness(new_solution)

            if new_score < scores[i]:
                population[i] = new_solution
                scores[i] = new_score

        # Abandon worst nests
        count = max(1, int(pa * nests))
        worst = np.argsort(scores)[-count:]

        population[worst] = np.random.uniform(
            1, 20, count
        )

        scores[worst] = np.array([
            fitness(x) for x in population[worst]
        ])

        # Update best solution
        index = np.argmin(scores)

        if scores[index] < best_score:
            best = population[index]
            best_score = scores[index]

    return best


# Main program
np.random.seed(11)

best_time = improved_cuckoo_search()

maximum_speed = 15 / best_time

print("Robot Trajectory Optimization")
print("Distance:", distance)
print("Maximum Allowed Velocity:", max_velocity)
print("Optimal Travel Time:", round(best_time, 2))
print("Maximum Robot Velocity:", round(maximum_speed, 2))

if maximum_speed <= max_velocity:
    print("Velocity Constraint: Satisfied")
else:
    print("Velocity Constraint: Not Satisfied")


Robot Trajectory Optimization
Distance: 10
Maximum Allowed Velocity: 2
Optimal Travel Time: 7.5
Maximum Robot Velocity: 2.0
Velocity Constraint: Satisfied
